# ch06_1. PyTorch 모델 저장·불러오기·추론

## 개요

- `state_dict()`가 저장하는 내용을 이해합니다.
- `load_state_dict()`로 모델 가중치를 복원합니다.

## 공통 설정

데이터·모델·이미지 폴더 위치를 지정합니다.

In [1]:
from pathlib import Path

DATA_DIR = Path("data")
MODEL_DIR = Path("models")
PNG_DIR = Path("png")

DATA_DIR.mkdir(exist_ok=True)

MODEL_DIR.mkdir(exist_ok=True)
PNG_DIR.mkdir(exist_ok=True)

print("데이터 폴더:", DATA_DIR)
print("모델 폴더  :", MODEL_DIR)
print("이미지 폴더:", PNG_DIR)


데이터 폴더: data
모델 폴더  : models
이미지 폴더: png


## 공통 import

In [2]:
import json
import platform
import shutil
import sys
import time
from pathlib import Path
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torchvision.models import resnet18, ResNet18_Weights

### 1) 실행 장치 확인

```text
1. 실행 환경 확인
2. 모델 소스 선택
3. ResNet18 모델 준비
4. state_dict와 Checkpoint 저장
5. CPU에서 모델 복원
6. 저장 전후 출력 비교
7. 원하는 장치에서 추론
8. 실제 이미지와 배치 추론
9. 평균 추론 시간 측정
10. PC 배포 폴더와 ZIP 생성
```

- 학습 코드는 다시 작성하지 않습니다.
- 모델 생성 함수는 하나만 사용합니다.
- 단일 추론과 배치 추론은 같은 함수로 처리합니다.
- 배포에 필요하지 않은 Optimizer(옵티마이저, 가중치 갱신 알고리즘) 상태는 저장하지 않습니다.
- 복잡한 자동 분기보다 설정값을 명확하게 표시합니다.

| 저장 방식 | 저장 내용 | 사용 목적 |
|---|---|---|
| `state_dict` | Weight(웨이트, 가중치), Bias(바이어스, 편향), Buffer(버퍼, 계층 상태값) | 가장 기본적인 모델 저장 |
| Checkpoint(체크포인트) | `state_dict` + 모델 정보 + 클래스 + 전처리 | 재현 가능한 추론·배포 |
| 전체 모델 객체 | Python 모델 객체 전체 | 코드 경로 의존성이 커서 기본 방식으로 권장하지 않음 |

```text
학습 재개
→ 모델 상태 + Optimizer 상태 + Epoch 정보

추론 배포
→ 모델 상태 + 모델 구조 정보 + 전처리 + 클래스 정보
```

BatchNorm(배치 노멀라이제이션, 배치 정규화)의 `running_mean`, `running_var`도 `state_dict`에 포함됩니다.

```text
모델 파일
.pt 또는 .pth

Runtime(런타임, 실행 엔진)
PyTorch

계산 장치
CPU, CUDA GPU 또는 Apple MPS
```

In [3]:
torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Python 버전 :", sys.version.split()[0])

print("PyTorch 버전:", torch.__version__)

print("운영체제     :", platform.system())

print("실행 장치    :", device)

if device.type == "cuda":
    print("GPU 이름     :", torch.cuda.get_device_name(0))

Python 버전 : 3.12.3
PyTorch 버전: 2.13.0+cu130
운영체제     : Linux
실행 장치    : cuda
GPU 이름     : NVIDIA GeForce RTX 3070 Laptop GPU


### 2) 모델 선택 설정

```text
ResNet18_Weights.DEFAULT
→ ImageNet-1K 사전학습 가중치
→ 1,000개 클래스
→ 별도 학습 없이 저장·복원·추론 실습
```

```text
CIFAR-10 전이학습 Checkpoint
→ CIFAR-10 10개 클래스
→ 마지막 fc 계층을 10개 출력으로 복원
→ 학습 결과를 이어서 사용
```

아래 설정에서 `MODEL_SOURCE`만 변경합니다.

```python
MODEL_SOURCE = "imagenet"
MODEL_SOURCE = "transfer"
```

In [4]:
MODEL_SOURCE = "transfer"

TRANSFER_CHECKPOINT_PATH = MODEL_DIR / "ch05_2_best_resnet18_transfer_learning.pt"

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

weights = ResNet18_Weights.DEFAULT

preprocess = weights.transforms()

### 3) ResNet18 생성 함수

In [5]:
def create_resnet18(num_classes=1000, pretrained=False):
    """출력 클래스 수에 맞는 ResNet18을 생성합니다."""
    selected_weights = weights if pretrained else None
    model = resnet18(weights=selected_weights)

    if num_classes != 1000:
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

### 4) 모델 준비

In [6]:
if MODEL_SOURCE == "transfer" and TRANSFER_CHECKPOINT_PATH.exists():
    source_checkpoint = torch.load(
        TRANSFER_CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True,
    )

    num_classes = int(
        source_checkpoint.get("number_of_classes", 10)
    )
    class_names = source_checkpoint.get(
        "class_names", CIFAR10_CLASSES
    )

    model = create_resnet18(num_classes=num_classes)
    model.load_state_dict(source_checkpoint["model_state_dict"])

    model_description = "CIFAR-10 Fine-Tuned ResNet18"
    dataset_name = "CIFAR-10"

elif MODEL_SOURCE == "transfer":
    # 전이학습 Checkpoint가 없어도 저장·복원 실습 자체는 실행할 수 있습니다.
    source_checkpoint = None
    num_classes = 10
    class_names = CIFAR10_CLASSES
    model = create_resnet18(num_classes=num_classes)

    model_description = "Demo ResNet18 (untrained)"
    dataset_name = "CIFAR-10 class layout"

    print("전이학습 Checkpoint가 없어 실습용 미학습 ResNet18을 사용합니다.")

else:
    source_checkpoint = None
    num_classes = 1000
    class_names = list(weights.meta["categories"])

    try:
        model = create_resnet18(
            num_classes=num_classes,
            pretrained=True,
        )
        model_description = "ImageNet Pretrained ResNet18"
    except Exception:
        model = create_resnet18(
            num_classes=num_classes,
            pretrained=False,
        )
        model_description = "Demo ResNet18 (untrained)"
        print("사전학습 가중치를 받을 수 없어 미학습 ResNet18을 사용합니다.")

    dataset_name = "ImageNet-1K"


### 5) 모델 확인

In [7]:
model = model.to(device)

model.eval()

print("모델       :", model_description)

print("데이터 기준:", dataset_name)

print("클래스 수  :", num_classes)

모델       : CIFAR-10 Fine-Tuned ResNet18
데이터 기준: CIFAR-10
클래스 수  : 10


### 6) state_dict 구조와 기준 출력

`state_dict()`는 다음과 같은 Dictionary(딕셔너리, 키와 값의 묶음)입니다.

```text
Key
→ 계층과 파라미터 이름

Value
→ 실제 Tensor(텐서, 다차원 수치 배열)
```

저장 전 기준 출력은 다음과 같습니다.

\[
\mathbf{y}_{ref}=f_{\theta}(\mathbf{x})
\]

- $\mathbf{x}$: 기준 입력
- $f_{\theta}$: 가중치 $\theta$를 가진 모델
- $\mathbf{y}_{ref}$: 저장 전 모델의 기준 출력

In [8]:
model_state = model.state_dict()

print("state_dict 항목 수:", len(model_state))

print("\n앞쪽 8개 항목")

for index, (name, tensor) in enumerate(model_state.items()):
    print(f"{name:35s} → {tuple(tensor.shape)}")
    if index == 7:
        break

sample_input = torch.randn(1, 3, 224, 224, device=device)

with torch.inference_mode():
    reference_output = model(sample_input)

print("\n입력 Shape:", tuple(sample_input.shape))

print("출력 Shape:", tuple(reference_output.shape))

print("예측 클래스:", reference_output.argmax(dim=1).tolist())

state_dict 항목 수: 122

앞쪽 8개 항목
conv1.weight                        → (64, 3, 7, 7)
bn1.weight                          → (64,)
bn1.bias                            → (64,)
bn1.running_mean                    → (64,)
bn1.running_var                     → (64,)
bn1.num_batches_tracked             → ()
layer1.0.conv1.weight               → (64, 64, 3, 3)
layer1.0.bn1.weight                 → (64,)

입력 Shape: (1, 3, 224, 224)
출력 Shape: (1, 10)
예측 클래스: [4]


### 7) 모델 저장

```python
torch.save(model.state_dict(), "model_weights.pth")
```

```python
checkpoint = {
 "model_state_dict": model.state_dict(),
 "number_of_classes": num_classes,
 "class_names": class_names,
 "preprocessing": {...},
}
```

Checkpoint에는 모델을 다시 학습하지 않고 추론하기 위해 필요한 정보를 함께 넣습니다.

In [9]:
output_dir = MODEL_DIR / "ch06_1_pytorch_deployment"

output_dir.mkdir(exist_ok=True)

state_dict_path = output_dir / "resnet18_state_dict.pth"

checkpoint_path = output_dir / "resnet18_inference_checkpoint.pt"

torch.save(model.state_dict(), state_dict_path)

checkpoint = {
    "model_name": "resnet18",
    "model_description": model_description,
    "dataset_name": dataset_name,
    "number_of_classes": num_classes,
    "class_names": class_names,
    "model_state_dict": model.state_dict(),
    "input_shape": [1, 3, 224, 224],
    "input_dtype": "float32",
    "tensor_layout": "NCHW",
    "preprocessing": {
        "resize": 256,
        "center_crop": 224,
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
    },
}

torch.save(checkpoint, checkpoint_path)

print("가중치 파일    :", state_dict_path)

print("Checkpoint 파일:", checkpoint_path)

가중치 파일    : models/ch06_1_pytorch_deployment/resnet18_state_dict.pth
Checkpoint 파일: models/ch06_1_pytorch_deployment/resnet18_inference_checkpoint.pt


### 8) 모델 복원

```text
1. Checkpoint를 CPU로 불러오기
2. 저장된 클래스 수 확인
3. 같은 모델 구조 생성
4. load_state_dict()로 가중치 적용
5. model.eval()로 추론 모드 전환
```

`map_location="cpu"`를 사용하면 GPU에서 저장한 Tensor도 먼저 CPU로 불러올 수 있습니다.

In [10]:
loaded_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True,
)

restored_model = create_resnet18(
    num_classes=int(loaded_checkpoint["number_of_classes"])
)

restored_model.load_state_dict(
    loaded_checkpoint["model_state_dict"]
)

restored_model.eval()

print("복원 장치   :", next(restored_model.parameters()).device)

print("출력 클래스 :", restored_model.fc.out_features)

print("학습 모드   :", restored_model.training)

복원 장치   : cpu
출력 클래스 : 10
학습 모드   : False


### 9) 저장·복원 결과 비교

복원한 모델 출력:

\[
\mathbf{y}_{load}=f_{\hat{\theta}}(\mathbf{x})
\]

최대 절대 오차:

\[
E_{\max}
=
\max_i
\left|
y_{ref,i}-y_{load,i}
\right|
\]

가중치가 정확히 복원되면 $E_{\max}$는 0 또는 매우 작은 값입니다.

In [ ]:
# 검증용 입력을 CPU로 이동합니다.
cpu_input = sample_input.detach().cpu()

# 원본 모델과 동일한 구조의 CPU 모델을 만듭니다.
reference_model_cpu = create_resnet18(
    num_classes=num_classes
)

# 저장 직전 원본 모델의 가중치를 CPU 모델에 복사합니다.
reference_model_cpu.load_state_dict(
    model.state_dict()
)

reference_model_cpu.eval()

# 원본 가중치와 복원 가중치를 
# 동일한 CPU 환경에서 각각 추론합니다.
with torch.inference_mode():
    reference_cpu = reference_model_cpu(cpu_input)
    restored_output = restored_model(cpu_input)

# 두 결과의 최대 차이를 확인합니다.
max_error = (
    reference_cpu - restored_output
).abs().max().item()

# 같은 장치에서 동일한 가중치로 계산했으므로
# 엄격한 기준으로 비교할 수 있습니다.
torch.testing.assert_close(
    reference_cpu,
    restored_output,
    rtol=1e-5,
    atol=1e-6,
)

print(f"최대 절대 오차: {max_error:.8f}")
print("저장·복원 검증: 통과")

최대 절대 오차: 0.00000000
저장·복원 검증: 통과


### 10) 추론 함수 정의

- Dropout(드롭아웃, 일부 뉴런을 임시로 끄는 정규화)을 추론 방식으로 전환합니다.
- BatchNorm의 이동 평균과 이동 분산을 사용합니다.

- Gradient(그래디언트, 기울기) 계산을 중지합니다.
- 추론에 필요하지 않은 계산 그래프 생성을 줄입니다.

```python
model.eval()

with torch.inference_mode():
 output = model(input_tensor)
```

두 기능은 역할이 다르므로 추론할 때 함께 사용합니다.

분류 모델은 Logit(로짓, 확률 변환 전 점수)을 출력합니다.

\[
p_i
=
\frac{e^{z_i}}
{\sum_{j=1}^{C}e^{z_j}}
\]

- $z_i$: $i$번째 클래스의 Logit
- $p_i$: $i$번째 클래스의 예측 확률
- $C$: 전체 클래스 수

단일 이미지는 `(C, H, W)`, 배치 이미지는 `(N, C, H, W)` Shape를 사용합니다.

In [13]:
def predict(model, input_tensor, class_names, device, top_k=5):
    model = model.to(device).eval()

    if input_tensor.ndim == 3:
        input_tensor = input_tensor.unsqueeze(0)

    input_tensor = input_tensor.to(device)

    with torch.inference_mode():
        logits = model(input_tensor)
        probabilities = torch.softmax(logits, dim=1)
        top_probs, top_indices = probabilities.topk(
            k=min(top_k, probabilities.shape[1]),
            dim=1,
        )

    batch_results = []

    for sample_probs, sample_indices in zip(
        top_probs.cpu(), top_indices.cpu()
    ):
        sample_results = []

        for probability, class_index in zip(
            sample_probs, sample_indices
        ):
            index = int(class_index)
            sample_results.append({
                "class_index": index,
                "class_name": class_names[index],
                "probability": float(probability),
            })

        batch_results.append(sample_results)

    return batch_results

### 11) 단일 입력 추론

In [14]:
single_results = predict(
    restored_model,
    cpu_input[0],
    loaded_checkpoint["class_names"],
    device,
    top_k=5,
)[0]

print("===== 단일 Tensor Top-5 =====")

===== 단일 Tensor Top-5 =====


### 12) Top-5 결과 출력

In [15]:
for rank, result in enumerate(single_results, start=1):
    print(
        f"{rank}. {result['class_name']}"
        f" - {result['probability'] * 100:.2f}%"
    )

1. deer - 49.80%
2. bird - 21.60%
3. airplane - 14.71%
4. frog - 7.91%
5. ship - 2.70%


### 13) 배치 입력 추론

In [16]:
batch_input = torch.randn(4, 3, 224, 224)

batch_results = predict(
    restored_model,
    batch_input,
    loaded_checkpoint["class_names"],
    device,
    top_k=1,
)

print("\n===== 배치 Tensor Top-1 =====")


===== 배치 Tensor Top-1 =====


### 14) 배치 결과 출력

In [17]:
for index, result in enumerate(batch_results):
    prediction = result[0]
    print(
        f"{index}: {prediction['class_name']}"
        f" - {prediction['probability'] * 100:.2f}%"
    )

0: deer - 60.31%
1: deer - 53.80%
2: deer - 55.22%
3: deer - 52.94%


### 15) 실제 이미지 추론

```text
이미지 열기
→ RGB 변환
→ Resize·Crop·Normalize
→ Batch 차원 추가
→ 모델 추론
→ Top-k 결과 출력
```

ImageNet 사전학습 ResNet18은 `ResNet18_Weights.DEFAULT.transforms()`가 제공하는 전처리를 사용합니다.

In [18]:
IMAGE_PATH = None

if IMAGE_PATH is not None:
    image = Image.open(IMAGE_PATH).convert("RGB")
    image_tensor = preprocess(image)

    image_results = predict(
        restored_model,
        image_tensor,
        loaded_checkpoint["class_names"],
        device,
        top_k=5,
    )[0]

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title(IMAGE_PATH.name)
    plt.show()

    print("===== 실제 이미지 Top-5 =====")
    for rank, result in enumerate(image_results, start=1):
        print(
            f"{rank}. {result['class_name']}"
            f" - {result['probability'] * 100:.2f}%"
        )
else:
    print("IMAGE_PATH에 이미지 파일 경로를 지정하면 실행됩니다.")

IMAGE_PATH에 이미지 파일 경로를 지정하면 실행됩니다.


### 16) 평균 추론 시간 측정

\[
T_{latency}
=
\frac{t_{end}-t_{start}}{N}
\]

- $N$: 반복 추론 횟수
- $T_{latency}$: 한 번 추론하는 데 걸리는 평균 시간

GPU에서는 연산이 비동기로 실행될 수 있으므로 측정 전후에 동기화합니다.

In [19]:
def measure_latency(model, input_tensor, device, warmup=5, repeat=20):
    model = model.to(device).eval()
    input_tensor = input_tensor.to(device)

    with torch.inference_mode():
        for _ in range(warmup):
            model(input_tensor)

        if device.type == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        for _ in range(repeat):
            model(input_tensor)

        if device.type == "cuda":
            torch.cuda.synchronize()

    return (time.perf_counter() - start) / repeat

timing_input = torch.randn(1, 3, 224, 224)

average_seconds = measure_latency(
    restored_model,
    timing_input,
    device,
)

print("측정 장치      :", device)

print(f"평균 추론 시간: {average_seconds * 1000:.3f} ms")

측정 장치      : cuda
평균 추론 시간: 2.047 ms


### 17) 배포 파일 생성

```text
model checkpoint
→ 모델 가중치와 기본 정보

inference_config.json
→ 입력 Shape와 전처리 규칙

class_names.json
→ 출력 Index와 클래스 이름의 대응

sample_input.pt
→ 기준 입력

reference_output.pt
→ 기준 출력

inference_example.py
→ 독립 실행 예제
```

In [21]:
config = {
    "model_name": "resnet18",
    "runtime": "PyTorch",
    "input_shape": [1, 3, 224, 224],
    "input_dtype": "float32",
    "tensor_layout": "NCHW",
    "output_type": "logits",
    "number_of_classes": num_classes,
    "preprocessing": checkpoint["preprocessing"],
}

(output_dir / "inference_config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

(output_dir / "class_names.json").write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

saved_input = cpu_input[:1]

with torch.inference_mode():
    saved_output = restored_model.cpu()(saved_input)

torch.save(saved_input, output_dir / "sample_input.pt")

torch.save(saved_output, output_dir / "reference_output.pt")

inference_script = 'import torch\nimport torch.nn as nn\nfrom pathlib import Path\nfrom torchvision.models import resnet18\n\nbase_dir = Path(__file__).resolve().parent\n\ndevice = torch.device(\n    "cuda" if torch.cuda.is_available() else "cpu"\n)\n\ncheckpoint = torch.load(\n    base_dir / "resnet18_inference_checkpoint.pt",\n    map_location="cpu",\n    weights_only=True,\n)\n\nnum_classes = int(checkpoint["number_of_classes"])\n\nmodel = resnet18(weights=None)\n\nif num_classes != 1000:\n    model.fc = nn.Linear(model.fc.in_features, num_classes)\n\nmodel.load_state_dict(checkpoint["model_state_dict"])\nmodel = model.to(device).eval()\n\ninput_tensor = torch.load(\n    base_dir / "sample_input.pt",\n    map_location=device,\n    weights_only=True,\n)\n\nwith torch.inference_mode():\n    logits = model(input_tensor)\n    probabilities = torch.softmax(logits, dim=1)\n    confidence, prediction = probabilities.max(dim=1)\n\nclass_names = checkpoint["class_names"]\nprediction_index = int(prediction.item())\n\nprint("Prediction:", class_names[prediction_index])\nprint("Confidence:", f"{float(confidence.item()) * 100:.2f}%")\n'

(output_dir / "inference_example.py").write_text(
    inference_script,
    encoding="utf-8",
)

print("배포 파일 저장 완료")

배포 파일 저장 완료


### 18) 배포 폴더 확인과 ZIP 생성

```text
ch06_1_pytorch_deployment/
├─ resnet18_state_dict.pth
├─ resnet18_inference_checkpoint.pt
├─ inference_config.json
├─ class_names.json
├─ sample_input.pt
├─ reference_output.pt
└─ inference_example.py
```

In [22]:
print("===== 배포 폴더 =====")

for file_path in sorted(output_dir.iterdir()):
    size_kb = file_path.stat().st_size / 1024
    print(f"{file_path.name:40s} {size_kb:10.1f} KB")

zip_path = shutil.make_archive(
    base_name=str(output_dir),
    format="zip",
    root_dir=output_dir,
)

print("\nZIP 파일:", zip_path)

===== 배포 폴더 =====
class_names.json                                0.1 KB
inference_config.json                           0.4 KB
inference_example.py                            1.1 KB
reference_output.pt                             1.6 KB
resnet18_inference_checkpoint.pt            43759.1 KB
resnet18_state_dict.pth                     43757.1 KB
sample_input.pt                               589.6 KB

ZIP 파일: ~/lab_jupyter/lab_ML_DL_2/models/ch06_1_pytorch_deployment.zip
